# PLAUD: Performative Latents for Adaptive Unsupervised DDSP

**PLAUD** is a modular PyTorch framework for Differentiable Digital Signal Processing (DDSP). It targets small, personal datasets and emphasizes playability, modularity, and real-time use. The system is configurable end-to-end: datasets, feature/control spaces, synthesis blocks, losses, training regimes (including optional adversarial), and a transformer-based Prior model for generative control.

Sourcecode: https://github.com/blazejkotowski/PLAUD

-----

Notebook author: [Martin Heinze](https://codeberg.org/martstilde)

Last updated: 06.11.2026

## Setup

In [ ]:
#Install PLAUD
#Clones v2 branch from original repo; in case of errors due to major changes in the original repo, try the fork's v2 branch

!mkdir /kaggle/temp
%cd /kaggle/temp
!git clone -b v2 https://github.com/blazejkotowski/PLAUD/ 
#!git clone -b v2 https://github.com/devstermarts/plaud/ 
%cd /kaggle/temp/plaud
!pip install -r requirements.txt
!pip install -e .

## Data and config retrieval

In [ ]:
#Copy dataset and configuration to working folder.

!mkdir /kaggle/working/data
!cp -r /kaggle/input/your-audio-data/* /kaggle/working/data
!cp /kaggle/input/your-config.yaml /kaggle/working/your-config.yaml

In [ ]:
##Optional: copy previous run files to /kaggle/working folder to continue training from an earlier checkpoint.
#!cp -r /kaggle/input/your-earlier-run/* /kaggle/working

## Training

In [ ]:
#Train DDSP synthesizer

%cd /kaggle/working
!CUDA_VISIBLE_DEVICES=1 python -m cli.train -cd /kaggle/working -cn your-config

In [ ]:
#Train prior

%cd /kaggle/working
!CUDA_VISIBLE_DEVICES=1 python -m cli.train_prior -cd /kaggle/working -cn your-config

In [ ]:
#Train postnet

%cd /kaggle/working
!CUDA_VISIBLE_DEVICES=1 python -m cli.train_postnet -cd /kaggle/working -cn your-config \
++postnet.enabled=true

## Export

In [ ]:
#Export a streamable .ts model

%cd /kaggle/working
!CUDA_VISIBLE_DEVICES=1 python -m cli.export --config /kaggle/working/your-config.yaml \
--model_directory /kaggle/working/synth/your-config \
--compressor_checkpoint /kaggle/working/compressor/your-config/best.ckpt \
--prior_directory /kaggle/working/prior_discrete/your-config \
--postnet /kaggle/working/postnet/your-config/best.ckpt \
--output_path /kaggle/working/plaud_your-config.ts